In [ ]:
# Paths (relative to the repository root, so the notebooks run on any machine)
# Expected layout:  <repo>/notebooks/ (this file), <repo>/data/, <repo>/models/,
#                   <repo>/figures/, <repo>/results/, <repo>/feature_importance/
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
FIG_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
FI_DIR = ROOT / "feature_importance"
for _d in (DATA_DIR, MODELS_DIR, FIG_DIR, RESULTS_DIR, FI_DIR):
    _d.mkdir(parents=True, exist_ok=True)


## **Catboost Regression**

In [1]:
import pandas as pd
import numpy as np
import joblib

from catboost import CatBoostRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error
)

from scipy.stats import pearsonr, spearmanr

from sklearn.model_selection import RandomizedSearchCV, KFold

# recreate the evaluation function

def evaluate(y_true, pred, title):
    
    print(title)
    print("-"*30)

    print("R²   :", round(r2_score(y_true, pred),4))
    print("MAE  :", round(mean_absolute_error(y_true, pred),4))
    print("RMSE :", round(np.sqrt(mean_squared_error(y_true, pred)),4))
    print()

In [2]:
# Load the train/test data

X_train = pd.read_csv(
    str(DATA_DIR / "X_train1.csv")
)

X_test = pd.read_csv(
    str(DATA_DIR / "X_test1.csv")
)

y_train = pd.read_csv(
    str(DATA_DIR / "y_train1.csv")
).squeeze("columns")

y_test = pd.read_csv(
    str(DATA_DIR / "y_test1.csv")
).squeeze("columns")

In [3]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(21440, 1442)
(5361, 1442)
(21440,)
(5361,)


In [4]:
# Create the baseline CatBoost model

cat = CatBoostRegressor(
    random_state=42,
    verbose=0
)

In [5]:
# Train the CatBoost Model

cat.fit(X_train, y_train)

CatBoostRegressor(loss_function='RMSE', random_state=42, verbose=0)

In [6]:
# Make the predictions

train_pred = cat.predict(X_train)
test_pred = cat.predict(X_test)

In [7]:
# Evalute the model

evaluate(
    y_train,
    train_pred,
    "CatBoost Training"
)

evaluate(
    y_test,
    test_pred,
    "CatBoost Testing"
)

CatBoost Training
------------------------------
R²   : 0.8167
MAE  : 0.2612
RMSE : 0.3443

CatBoost Testing
------------------------------
R²   : 0.7259
MAE  : 0.3086
RMSE : 0.4105



In [8]:
# Save the Feature Importance

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": cat.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance.to_csv(
    str(FI_DIR / "CatBoost_FeatureImportance1.csv"),
    index=False
)

In [9]:
importance.head(20)

,Feature,Importance
0,Charge_complex,3.637186
106,MolLogP,2.291059
39,PEOE_VSA5,2.098093
613,FilterItLogS,1.912419
628,n6HRing,1.370746
313,AATSC0c,1.342156
166,fr_pyridine,1.323502
619,piPC9,1.297623
30,PEOE_VSA1,1.173380
31,PEOE_VSA10,1.168556


## **Hyperparameter Tuning**

In [10]:
# Define the parameter grid

param_dist = {
    "iterations": [300, 500, 700, 1000],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "depth": [4, 6, 8, 10],
    "l2_leaf_reg": [1, 3, 5, 7, 9],
    "bagging_temperature": [0, 0.5, 1, 3, 5],
    "random_strength": [1, 2, 5, 10]
}

In [11]:
# Create the CatBoost Model

cat = CatBoostRegressor(
    loss_function="RMSE",
    random_state=42,
    verbose=0,
)

In [12]:
# CV splitter

cv_splitter = KFold(n_splits = 5, shuffle = True, random_state = 42)

In [13]:
# Create RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=cat,
    param_distributions=param_dist,
    n_iter=30,
    scoring="r2",
    cv=cv_splitter,
    verbose=2,
    random_state=42,
    n_jobs=-2
)

In [14]:
# Start to tuning the CatBoost Model

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","CatBoostRegre...42, verbose=0)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'bagging_temperature': [0, 0.5, ...], 'depth': [4, 6, ...], 'iterations': [300, 500, ...], 'l2_leaf_reg': [1, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-2
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to 

In [15]:
# Check the best parameters

print (random_search.best_params_)

{'random_strength': 1, 'learning_rate': 0.1, 'l2_leaf_reg': 1, 'iterations': 1000, 'depth': 10, 'bagging_temperature': 1}


In [16]:
# Check the best cross validation score

print (random_search.best_score_)

0.783312510013289


In [17]:
# Retrieve the optimized CatBoost Model

best_cat = random_search.best_estimator_

In [18]:
# Make the predictions

train_pred = best_cat.predict(X_train)
test_pred = best_cat.predict(X_test)

In [19]:
# Evalute the optimized CatBoost Model

evaluate(
    y_train,
    train_pred,
    "Optimized CatBoost - Training"
)

evaluate(
    y_test,
    test_pred,
    "Optimized CatBoost - Testing"
)

Optimized CatBoost - Training
------------------------------
R²   : 0.945
MAE  : 0.1377
RMSE : 0.1885

Optimized CatBoost - Testing
------------------------------
R²   : 0.8031
MAE  : 0.251
RMSE : 0.3479



In [20]:
# Save the feature importance

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_cat.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

importance.to_csv(
    str(FI_DIR / "Optimize_CatBoost_FeatureImportance1.csv"),
    index=False
)

In [21]:
# Save the Optimize model

import joblib

joblib.dump(
    best_cat,
    str(MODELS_DIR / "Best_CatBoost_Model1.pkl")
)

['C:\\Users\\user\\Ph.d Work\\Models\\Best_CatBoost_Model1.pkl']

## **Calculate the Metrics**   

In [22]:
# Load the Train/Test Dataset

X_train = pd.read_csv(str(DATA_DIR / "X_train1.csv"))
X_test = pd.read_csv(str(DATA_DIR / "X_test1.csv"))

y_train = pd.read_csv(str(DATA_DIR / "y_train1.csv")).squeeze()
y_test = pd.read_csv(str(DATA_DIR / "y_test1.csv")).squeeze()

In [23]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(21440, 1442)
(5361, 1442)
(21440,)
(5361,)


In [24]:
# load the saved best model

best_model = joblib.load(str(MODELS_DIR / "Best_CatBoost_Model1.pkl"))

In [25]:
# Generate the prediction

y_train_pred = best_model.predict(X_train)

y_test_pred = best_model.predict(X_test)

In [26]:
# Calculate R²
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

In [27]:
# Calculate RMSE
train_rmse = root_mean_squared_error(y_train, y_train_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)

In [28]:
# Calculate MAE
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

In [29]:
# Calculate Pearson correlation

train_pearson, train_pearson_p = pearsonr(
    y_train,
    y_train_pred
)

test_pearson, test_pearson_p = pearsonr(
    y_test,
    y_test_pred
)

In [30]:
# Calculate Spearman correlation

train_spearman, train_spearman_p = spearmanr(
    y_train,
    y_train_pred
)

test_spearman, test_spearman_p = spearmanr(
    y_test,
    y_test_pred
)

In [31]:
# Create the manuscript table

results = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "R²": [train_r2, test_r2],
    "RMSE": [train_rmse, test_rmse],
    "MAE": [train_mae, test_mae],
    "Pearson r": [train_pearson, test_pearson],
    "Pearson p-value": [train_pearson_p, test_pearson_p],
    "Spearman ρ": [train_spearman, test_spearman],
    "Spearman p-value": [train_spearman_p, test_spearman_p]
})

results = results.round(4)

results

,Dataset,R²,RMSE,MAE,Pearson r,Pearson p-value,Spearman ρ,Spearman p-value
0,Training,0.9450,0.1885,0.1377,0.9736,0.0,0.9715,0.0
1,Testing,0.8031,0.3479,0.2510,0.8970,0.0,0.8926,0.0


In [32]:
# Save the table

results.to_csv(
    str(RESULTS_DIR / "CatBoost_Performance_Table1.csv"),
    index=False
)